# 5 · (Optional) ROUGE cross-check

Reference-based automatic metrics on the saved eval samples. These are **cheap and reproducible but weakly correlated with human-perceived quality** — use them as a sanity cross-check, NOT as the headline. The usual finding: RLAIF can *improve* judge win-rate while ROUGE stays flat (RL moves summaries away from the single reference wording). That gap is the whole reason we need the judge.

> **Dependency-free** — ROUGE-1/2/L are computed with stdlib (no `rouge-score`). The eval job already reports a ROUGE-L too; this adds ROUGE-1/2 per model.

In [ ]:
import json
samples = json.load(open('results/samples.json'))
models = list(samples[0]['generations'].keys())
print('models:', models, '| samples:', len(samples))

In [ ]:
def _f1(overlap, n_pred, n_ref):
    if n_pred == 0 or n_ref == 0 or overlap == 0:
        return 0.0
    p, r = overlap / n_pred, overlap / n_ref
    return 2 * p * r / (p + r)

def _ngrams(toks, n):
    return [tuple(toks[i:i+n]) for i in range(len(toks) - n + 1)]

def rouge_n(pred, ref, n):
    from collections import Counter
    p, r = _ngrams(pred.lower().split(), n), _ngrams(ref.lower().split(), n)
    if not p or not r:
        return 0.0
    overlap = sum((Counter(p) & Counter(r)).values())
    return _f1(overlap, len(p), len(r))

def rouge_l(pred, ref):
    a, b = pred.lower().split(), ref.lower().split()
    if not a or not b:
        return 0.0
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            dp[i][j] = dp[i-1][j-1]+1 if a[i-1]==b[j-1] else max(dp[i-1][j], dp[i][j-1])
    return _f1(dp[len(a)][len(b)], len(a), len(b))

In [ ]:
import numpy as np, pandas as pd
rows = []
for mdl in models:
    r1 = [rouge_n(s['generations'][mdl], s['reference'], 1) for s in samples]
    r2 = [rouge_n(s['generations'][mdl], s['reference'], 2) for s in samples]
    rl = [rouge_l(s['generations'][mdl], s['reference']) for s in samples]
    rows.append({'model': mdl, 'rouge1': np.mean(r1), 'rouge2': np.mean(r2), 'rougeL': np.mean(rl)})
display(pd.DataFrame(rows).set_index('model').round(4))
print('Reminder: ROUGE is a sanity check, not the headline. The LLM-judge pairwise win-rate is.')